# Step 5: Feature Engineering and Transformation

## Purpose
This notebook transforms the cleaned Telco churn dataset into model-ready features.

## Step 5 questions
- What information should the model actually see?
- Which columns should be excluded?
- How should categorical variables be encoded?
- Should numeric variables be scaled?
- How do we create `X` and `y` clearly and reproducibly?

## Expected outputs
- Feature matrix `X`
- Target vector `y`
- Encoded feature set for modeling
- Clear feature definitions and transformation logic

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
PROJECT_DIR = Path.cwd()
CLEANED_FILE = PROJECT_DIR / "WA_Fn-UseC_-Telco-Customer-Churn-cleaned.csv"

df = pd.read_csv(CLEANED_FILE)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Load the cleaned dataset

We begin from the cleaned CSV so feature engineering is built on consistent data rather than raw data.

In [3]:
df.shape, df.columns.tolist()

((7032, 21),
 ['customerID',
  'gender',
  'SeniorCitizen',
  'Partner',
  'Dependents',
  'tenure',
  'PhoneService',
  'MultipleLines',
  'InternetService',
  'OnlineSecurity',
  'OnlineBackup',
  'DeviceProtection',
  'TechSupport',
  'StreamingTV',
  'StreamingMovies',
  'Contract',
  'PaperlessBilling',
  'PaymentMethod',
  'MonthlyCharges',
  'TotalCharges',
  'Churn'])

## 2. Create a few domain-driven features

Before separating features and target, we add a few simple churn-relevant features that are still easy to interpret.

These are not complex features. They are practical and explainable:

- number of subscribed services
- average charge per month of tenure
- whether the customer is new
- whether the customer has a long contract

In [4]:
df['service_count'] = (
    (df['PhoneService'] == 'Yes').astype(int)
    + (df['MultipleLines'] == 'Yes').astype(int)
    + (df['InternetService'] != 'No').astype(int)
    + (df['OnlineSecurity'] == 'Yes').astype(int)
    + (df['OnlineBackup'] == 'Yes').astype(int)
    + (df['DeviceProtection'] == 'Yes').astype(int)
    + (df['TechSupport'] == 'Yes').astype(int)
    + (df['StreamingTV'] == 'Yes').astype(int)
    + (df['StreamingMovies'] == 'Yes').astype(int)
)

df['avg_monthly_value_from_total'] = (df['TotalCharges'] / df['tenure'].replace(0, 1)).round(2)
df['is_new_customer'] = (df['tenure'] <= 12).astype(int)
df['has_long_term_contract'] = df['Contract'].isin(['One year', 'Two year']).astype(int)

df[['service_count', 'avg_monthly_value_from_total', 'is_new_customer', 'has_long_term_contract']].head()

,service_count,avg_monthly_value_from_total,is_new_customer,has_long_term_contract
0,2,29.85,1,0
1,4,55.57,0,1
2,4,54.08,1,0
3,4,40.91,0,1
4,2,75.82,1,0


## 2. Define the target and exclude non-feature columns

For modeling, we exclude identifier columns and separate the churn label from the predictors.

In [5]:
target_column = 'Churn'
identifier_columns = ['customerID']

X = df.drop(columns=identifier_columns + [target_column]).copy()
y = df[target_column].map({'No': 0, 'Yes': 1}).copy()

print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)
print()
print('Target distribution:')
print(y.value_counts())

Feature matrix shape: (7032, 23)
Target vector shape: (7032,)

Target distribution:
Churn
0    5163
1    1869
Name: count, dtype: int64


## 3. Identify categorical and numeric features

We separate feature types because categorical columns need encoding, while numeric columns may need scaling depending on the model.

In [6]:
categorical_features = X.select_dtypes(include='object').columns.tolist()
numeric_features = X.select_dtypes(exclude='object').columns.tolist()

print('Categorical features:')
print(categorical_features)
print()
print('Numeric features:')
print(numeric_features)

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numeric features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'service_count', 'avg_monthly_value_from_total', 'is_new_customer', 'has_long_term_contract']


## 4. Encode and scale features

For this project:

- categorical variables will be one-hot encoded
- numeric variables will be scaled for model families that are sensitive to magnitude

This keeps the dataset usable for models like logistic regression while remaining interpretable.

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ]
)

X_transformed = preprocessor.fit_transform(X)

print('Transformed feature matrix shape:', X_transformed.shape)

Transformed feature matrix shape: (7032, 49)


In [8]:
feature_names = preprocessor.get_feature_names_out()
feature_names[:20]

array(['numeric__SeniorCitizen', 'numeric__tenure',
       'numeric__MonthlyCharges', 'numeric__TotalCharges',
       'numeric__service_count', 'numeric__avg_monthly_value_from_total',
       'numeric__is_new_customer', 'numeric__has_long_term_contract',
       'categorical__gender_Female', 'categorical__gender_Male',
       'categorical__Partner_No', 'categorical__Partner_Yes',
       'categorical__Dependents_No', 'categorical__Dependents_Yes',
       'categorical__PhoneService_No', 'categorical__PhoneService_Yes',
       'categorical__MultipleLines_No',
       'categorical__MultipleLines_No phone service',
       'categorical__MultipleLines_Yes',
       'categorical__InternetService_DSL'], dtype=object)

In [9]:
X_transformed_df = pd.DataFrame(X_transformed, columns=feature_names)
X_transformed_df.head()

,numeric__SeniorCitizen,numeric__tenure,numeric__MonthlyCharges,numeric__TotalCharges,numeric__service_count,numeric__avg_monthly_value_from_total,numeric__is_new_customer,numeric__has_long_term_contract,categorical__gender_Female,categorical__gender_Male,...,categorical__StreamingMovies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__PaperlessBilling_No,categorical__PaperlessBilling_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check
0,-0.440327,-1.280248,-1.161694,-0.994194,-0.928661,-1.157887,1.494357,-0.902613,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,-0.440327,0.064303,-0.260878,-0.173740,-0.063657,-0.305773,-0.669184,1.107895,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,-0.440327,-1.239504,-0.363923,-0.959649,-0.063657,-0.355138,1.494357,-0.902613,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,-0.440327,0.512486,-0.747850,-0.195248,-0.063657,-0.791465,-0.669184,1.107895,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,-0.440327,-1.239504,0.196178,-0.940457,-0.928661,0.365117,1.494357,-0.902613,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


## 5. Feature engineering notes

This feature set is intentionally simple and interpretable:

- `customerID` is excluded because it is only an identifier
- `Churn` is converted to a binary target
- categorical variables are one-hot encoded
- numeric variables are scaled for models like logistic regression
- a few domain-driven features were added:
  - `service_count`
  - `avg_monthly_value_from_total`
  - `is_new_customer`
  - `has_long_term_contract`

This gives us a model-ready feature set without making interpretation difficult.